# 사전 설정 및 특성값 공학

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np

In [ ]:
CSV_PATH = "inbody_dataset.csv"

# 시스템에 설치된 폰트 중에서 한글 폰트 찾기
font_list = [f.name for f in fm.fontManager.ttflist]
korean_fonts = [f for f in font_list if 'Gothic' in f or 'Malgun' in f]

if korean_fonts:
    plt.rcParams['font.family'] = korean_fonts[0]
    print(f"사용 폰트: {korean_fonts[0]}")
else:
    print("한글 폰트를 찾을 수 없습니다.")

plt.rcParams['axes.unicode_minus'] = False

## 데이터 로드

In [ ]:
df = pd.read_csv(CSV_PATH, encoding="utf-8")
df['날짜'] = pd.to_datetime(df['날짜'], errors='coerce')

numeric_columns = [
    '키', '골격근량', '체수분', '단백질',
    '무기질', '체지방', '체중',
    '복부지방률', '내장지방레벨'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

df = df.dropna(subset=['날짜'])
df = df.sort_values('날짜').reset_index(drop=True)

print(df.head())
print("\n=== 결측치 확인 ===")
print(df.isnull().sum())

## 특성값 공학

In [ ]:
df['BMI'] = df['체중'] / ((df['키'] / 100) ** 2)
df['체지방률'] = df['체지방'] / df['체중'] * 100

df['날짜'] = (df['날짜'] - df['날짜'].iloc[0]).dt.days

In [ ]:
df.head()

## 통계 분석

In [ ]:
# 최초 대비 최근 변화량
change_columns = [
    '체중', '골격근량', '체지방',
    '체지방률', 'BMI', '복부지방률',
    '내장지방레벨'
]

print("최초 대비 최근 변화량")

for col in change_columns:
    first = df[col].iloc[0]
    last = df[col].iloc[-1]

    if pd.notna(first) and pd.notna(last):
        print(f"{col}: {first:.2f} -> {last:.2f} (변화: {last - first:+.2f})")

# 시각화

## 헬퍼함수

In [ ]:
# 헬퍼함수
def paint(df, column, axes, positionX, positionY):
    ax = axes[positionX, positionY]

    ax.plot(
        df['날짜'],
        df[column],
        marker='o',
        label=column
    )
    
    ax.legend()
    ax.grid(True, alpha=0.3)

In [ ]:
def paint(df, columns, axes, positionX, positionY):
    ax = axes[positionX, positionY]
    x = df['날짜']
    for column in columns:
        ax.plot(
            x,
            df[column],
            marker='o',
            linewidth=1.5,
            label=column
        )

    ax.set_xticks(x)
    ax.set_xticklabels([xi for xi in x], rotation=0, fontsize=8)
    
    y = np.concatenate([df[column].values for column in columns])
    y = np.unique(y)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{yi:,.1f}" for yi in y], fontsize=8)
    
        
    ax.legend()
    ax.grid(True, alpha=0.3)

## 시각화코드

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)

# 체중
paint(df, ['체중'], axes, 0, 0)

# BMI
paint(df, ['BMI'], axes, 0, 1)

# 복부지방률
paint(df, ['복부지방률'], axes, 0, 2)

# 골격근량 + 체지방
paint(df, ['골격근량', '체지방'], axes, 1, 0)

# 체지방률
paint(df, ['체지방률'], axes, 1, 1)

# 내장지방레벨
paint(df, ['내장지방레벨'], axes, 1, 2)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(5,4), squeeze=False)

paint(df, ['체수분', '단백질', '무기질', '체지방'], axes, 0, 0)

plt.tight_layout()
plt.show()

In [ ]:
correlation_columns = [
    '체중', '골격근량', '체수분', '단백질',
    '무기질', '체지방',
    '복부지방률', '내장지방레벨'
]

corr = df[correlation_columns].corr(method='pearson')

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title('체성분 변수 간 상관관계')
plt.tight_layout()
plt.show()